# 🧪 MER-Lab: Multimodal Emotion Recognition on MELD
## Complete Experimental Suite for Research Hypotheses H1–H7

This notebook trains and benchmarks the **Trimodal Dynamic Gated Cross-Attention (DGCA)** architecture against unimodal, bimodal, and baseline fusion models on the **real MELD** dataset (Text, Audio, Video).

### Evaluated Hypotheses:
1. **H1 — Multimodal Superiority**: Trimodal (Text+Audio+Video) vs Unimodal (T, A, V) & Bimodal
2. **H2 — Advanced Fusion**: Proposed DGCA vs Concat, Average, and Self-Attention
3. **H3 — Dynamic Modality Gating**: Input-conditioned adaptive modality weighting
4. **H4 — Missing-Modality Robustness**: Inference performance under dropped modalities
5. **H5 — Lightweight Efficiency**: Fast training on frozen foundation representations
6. **H6 — Component Ablations**: Impact of Attention, Gating, and Modality Dropout
7. **H7 — Class-Level Analysis**: Disproportionate benefits for minority emotions (*fear*, *disgust*)

---

### Step 1: GPU Environment Check & Dependencies

In [ ]:
# Check GPU Availability
!nvidia-smi

# Install required packages (100% free, open-source)
!pip install -q transformers pyyaml torch tqdm matplotlib seaborn scikit-learn

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Clone / Setup MER-Lab Repository
If running in Colab, run the cell below to fetch or update the latest codebase:

In [ ]:
import os
import sys
from pathlib import Path

# If not already inside MER-Lab directory, clone; if already inside, pull latest
if not os.path.exists("src"):
    !git clone https://github.com/UtsavChandegara/MER-Lab.git
    %cd MER-Lab
else:
    !git pull origin main

sys.path.insert(0, os.path.abspath("."))
print(f"Current working directory: {os.getcwd()}")

### Step 3: Real MELD Dataset Acquisition & RoBERTa Feature Extraction
Downloads official MELD annotations and extracts real 768-dim RoBERTa embeddings (100% free).

In [ ]:
# Run automated MELD data preparation script
!python scripts/prepare_meld_data.py --output_dir data/meld

import torch
from pathlib import Path

data_dir = Path("data/meld")
train_data = torch.load(data_dir / "train_features.pt")
print(f"\nSuccessfully Loaded Real MELD Dataset!")
print(f"Train Samples: {len(train_data['labels']):,}")
print(f"Text Shape:    {train_data['text'].shape} (RoBERTa 768d)")
print(f"Audio Shape:   {train_data['audio'].shape} (Acoustic 768d)")
print(f"Video Shape:   {train_data['video'].shape} (Visual 512d)")
print(f"Sample Utterance 0: '{train_data['utterances'][0]}'")
print(f"Sample Emotion Label 0: {train_data['labels'][0].item()}")

### Step 4: Model Architecture & Trimodal Pipeline Check

In [ ]:
from src.foundation.config import Config
from src.ai_engine.builders.builder import ModelBuilder

config = Config.from_yaml("configs/meld_trimodal.yaml")
# Update dataset path and sample count to use the full MELD dataset
config._data["dataset"]["name"] = "meld_features"
config._data["dataset"]["data_dir"] = "data/meld"
config._data["dataset"]["num_samples"] = 9989

model = ModelBuilder.build_model(config)
print(model)

# Count trainable parameters (H5 - Lightweight Efficiency)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters in Proposed DGCA Model: {total_params:,}")
print("Notice: Entire fusion & classification network is ~1.5M parameters (Ultra-lightweight!)")

### Step 5: Execute Complete Hypothesis Benchmark Suite (H1–H7)
Runs all trials (Unimodal vs Multimodal, Fusion comparisons, Missing-modality stress tests, Ablations, and Class-level analyses).

In [ ]:
from src.research.experiments.benchmark_runner import BenchmarkSuite

# Initialize and run benchmark suite on GPU with real MELD data
suite = BenchmarkSuite(
    base_config_path="configs/meld_trimodal.yaml",
    output_dir="outputs/colab_benchmarks"
)
# Configure to use full real MELD data
suite.base_config._data["dataset"]["num_samples"] = 9989
suite.base_config._data["training"]["epochs"] = 10

results = suite.run_all()

### Step 6: Publication Figures & Plots
Generates publication-quality charts for the research paper (PNG and vector PDF at 300 DPI).

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

figures_dir = Path("outputs/colab_benchmarks/figures")
fig_files = sorted(figures_dir.glob("*.png"))

for fig_path in fig_files:
    img = Image.open(fig_path)
    plt.figure(figsize=(10, 5), dpi=150)
    plt.imshow(img)
    plt.axis("off")
    plt.title(fig_path.stem.replace("_", " ").title(), fontsize=12, fontweight="bold")
    plt.show()

### Step 7: LaTeX Tables Ready for Paper Insertion
Copy and paste these tables directly into your LaTeX manuscript.

In [ ]:
from pathlib import Path

latex_dir = Path("outputs/colab_benchmarks/latex_tables")
for tex_file in sorted(latex_dir.glob("*.tex")):
    print(f"\n{'='*20} {tex_file.name} {'='*20}\n")
    print(tex_file.read_text())

### Step 8: Download Complete Research Results (Zip)

In [ ]:
!zip -r meld_research_artifacts.zip outputs/colab_benchmarks/

try:
    from google.colab import files
    files.download("meld_research_artifacts.zip")
    print("Artifacts zip successfully downloaded!")
except ImportError:
    print("Saved artifacts to 'meld_research_artifacts.zip'.")